# Arrays and Hash Tables

Python reference-type semantics as they apply to arrays/lists: mutation vs. rebinding, `+=` vs `+`, `==` vs `is`, passing lists to functions, mutable vs. immutable types, and variable scope (LEGB).

## Reference types

A variable holding a list doesn't hold the list itself — it holds a reference (pointer) to where the list lives in memory.

```python
a = [1, 2, 3]
b = a
b.append(4)
print(a)  # ?
```

`a` and `b` point to the *same* list, so mutating through `b` is visible through `a` too.

Contrast with value types (ints, strings, tuples) — assignment copies the value, so mutating one name never affects the other.

In [ ]:
a = [1, 2, 3]
b = a
c = [1, 2, 3]
b.append(4)
print(a,b,c)

b = b + [4]
print(a, b, c)

[1, 2, 3, 4] [1, 2, 3, 4] [1, 2, 3]


## `+=` vs `+` for lists — mutation vs. new object

They look equivalent but aren't, for mutable types like lists:

```python
b += [4]      # calls __iadd__ -> mutates the list in place (like .extend())
b = b + [4]   # calls __add__   -> builds a brand-new list, then rebinds b
```

- `b += [4]`: mutates the existing list object. Any other name pointing at the same list (e.g. `a`) sees the change too.
- `b = b + [4]`: creates a new list at a new memory address, and rebinds `b` to point there. The connection between `a` and `b` is broken — `a` still points to the old address, `b` now points elsewhere. They're no longer aliases of the same object.

For immutable types (int, str, tuple), there's no in-place version, so `+=` and `x = x + y` behave identically.

**Interview signal:** if a question says an operation happens "in place," that means mutation — the original object is modified directly, no new object is created.

In [9]:
a = [1, 2, 3]
b = a
c = [1, 2, 3]
print(b == c)
print(a is b)

True
True


## `==` vs `is`, and Python vs Java

- `==` checks **value equality** — do the two objects contain the same data?
- `is` checks **identity** — are the two names pointing to the exact same object in memory (same address)?

```python
a = [1, 2, 3]
b = a
c = [1, 2, 3]

a is b   # True  -> same object (b was assigned from a)
b is c   # False -> different objects, even though...
b == c   # True  -> ...their contents are equal
```

**Python vs Java:** in Python, `==` on lists compares values by default (element-wise). In Java, `==` on arrays compares references (identity) — two arrays with identical elements are `==` false unless they're literally the same object; you need `Arrays.equals()` to compare values. So Python's default list equality is what Java makes you opt into explicitly.

In [11]:
def append_four(x):
    x.append(4)

def add_four(x):
    x = x + [4]

'''
a = [1, 2, 3]
append_four(a)
print(a)  

'''
b = [1, 2, 3]
add_four(b)
print(b) 


[1, 2, 3]


## Passing lists to functions

A function parameter is just another name pointing at the same object the caller passed in — same idea as `b = a`.

- `append_four(a)`: mutates the list in place (`x.append(4)`), so `a` shows the change after the call.
- `add_four(b)`: rebinds the *local* name `x` to a new list (`x = x + [4]`); that only affects `x`'s local frame, so `b` is unchanged after the call.

Rule of thumb: mutating methods on the parameter affect the caller; reassigning the parameter does not.

## Mutable vs. immutable — the real thing that bites people

Everything in Python is a reference — including "value-like" things such as ints. What actually differs is whether the *object* can be changed in place:

- **Mutable** (list, dict, set, most custom class instances): two names can share one object; mutating through one name is visible through the other.
- **Immutable** (int, str, tuple, float, bool, frozenset): the object itself can never change. `x = 6` after `x = 5` doesn't mutate the int `5` — it rebinds `x` to a different object. This *looks* like value semantics, but it's still references underneath; there's just no in-place mutation to observe.

Caveat: tuples are only shallowly immutable — you can't reassign an element, but if a tuple holds a mutable object (e.g. a list), that inner object can still be mutated.

### `x += 5` vs `x += [5]`

```python
x = 5
x += 5   # int has no __iadd__ -> falls back to __add__, creates a NEW int object, rebinds x

x = [5]
x += [5]  # list HAS __iadd__ -> mutates the list in place, same object, same address
```

`x += 5`: since ints are immutable, there's no in-place version — Python creates a new int object and rebinds `x` to it (new memory address). `x += [5]`: lists are mutable and define `__iadd__`, so this mutates the existing list in place (same address, same object as before).

In [15]:
x = "builtin"  # placeholder, not used directly

def outer():
    enclosing_var = "enclosing"

    def inner():
        local_var = "local"
        print(local_var)       # Local
        print(enclosing_var)   # Enclosing
        print(global_var)      # Global
        print(len)             # Builtin

    inner()

global_var = "global"
outer()

print("================")
#print(local_var)       # retuns error
#print(enclosing_var)   # returns error
print(global_var)      # Global
print(len)             # Builtin

local
enclosing
global
<built-in function len>
global
<built-in function len>


## Scope — LEGB

Scope is where a name is visible / which namespace it lives in. Python resolves names via the **LEGB** rule, checked in this order:

- **Local** — inside the current function
- **Enclosing** — an outer (enclosing) function, for nested functions/closures
- **Global** — module level
- **Builtin** — `len`, `print`, etc.

### Global vs. Builtin — why they're separate tiers

Each **module** (each `.py` file) has its own distinct global namespace — `mod_a.py` and `mod_b.py` do not share global variables. **Builtins**, by contrast, live in a single shared namespace (the `builtins` module) that every module implicitly consults — it is not duplicated per module.

If builtins were folded into "global," each module would need its own binding for `len`, `print`, etc., since global scope is module-local. Keeping Builtin as a separate, outermost tier means every module gets the same fallback definitions without redeclaring them, and any module can locally shadow a builtin name (e.g. defining its own `len`) without affecting other modules' access to the original.

## Implementing a dynamic array from scratch

Python's `list` is itself a dynamic array: fixed-size storage under the hood that reallocates (roughly doubles) when it fills up, giving `append` amortized O(1) time. Implementing a simplified version is a classic interview exercise and reinforces the reference/mutation concepts above.

In [51]:
# TODO: Implement a simplified dynamic array class `MyList` that:
# - starts empty, backed by a fixed-size internal array that doubles in capacity when full
# - supports append(value) -> amortized O(1)
# - supports len(mylist) -> number of elements currently stored (not the internal capacity)
# - supports mylist[i] -> get the i-th element
# - supports pop() -> remove and return the last element

class MyList:
    def __init__(self):
        self._length = 0
        self._lst = [None]
    
    def __len__(self):
        return self._length 
    
    def __getitem__(self, index):
        assert index < self._length, "index outside the length!"
        return self._lst[index]
    
    def pop(self):
        last_element = self._lst[self._length - 1]
        self._length -=1
        return last_element
    
    def _grow(self):
        _new_mem = [None] * self._length * 2
        for i in range(self._length):
            _new_mem[i] = self._lst[i]
        self._lst = _new_mem
    
    def append(self, value):
        self._lst[self._length] = value
        self._length +=1
        if len(self._lst) == self._length: 
            self._grow()
    
    def remove(self, index):
        for i in range(self._length):
            if i > index:
                self._lst[i - 1] = self._lst[i]
        self._length -= 1

In [68]:
# TODO: Reverse a string.

# solution 1: 

def reverse(string):
    list_str = list(string)
    n = len(list_str)

    new_list = [None] * n
    for i in range(n):
        new_list[i] = list_str[n-1-i]
    
    return ''.join(new_list)

# solution 2: the problem with solution 1 is the space complecity of o(n)

def reverse2(string):
    list_str = list(string)
    n = len(list_str)

    for i in range(n):
        if i < n//2:
            temp = list_str[i]
            list_str[i] = list_str[n-1-i]
            list_str[n-1-i] = temp
            
    return ''.join(list_str)

print(reverse2('aab'))

baa


In [ ]:
# TODO: Merge two sorted arrays into a single sorted array.

def merge(ls1, ls2):
    n = len(ls1)
    m = len(ls2)
    indx1 = 0
    indx2 = 0
    new_ls = [None] * (n+m)
    for i in range(n+m):
    
        if indx1 < n and indx2 < m: 
            if ls1[indx1] < ls2[indx2]:
                new_ls[i] = ls1[indx1]
                indx1 += 1 
            else:
                new_ls[i] = ls2[indx2]
                indx2 += 1
                
        elif indx1 < n:
            new_ls[i] = ls1[indx1]
            indx1 += 1

        else:
            new_ls[i] = ls2[indx2]
            indx2 += 1
            
    return new_ls

[7, 8, 10]
